# TanggapMakna V3 — Fusion Engine

## Final Candidate Architecture Before Blind Test

Notebook ini **tidak melakukan training**.

Input model:
1. **V3A Event Classifier**
   - sentence-level
   - `FLOOD / EARTHQUAKE / FIRE / ACCIDENT / NON_EVENT`

2. **V2 Semantic Role Labeling**
   - token-level
   - event roles + semantic arguments

Keduanya digabungkan oleh **deterministic Fusion / Consistency Rule Engine**.

### Locked architecture

`Free-text Report`
→ preprocessing
→ **V3A Event Classifier** + **V2 SRL Extractor** (parallel)
→ Fusion / Consistency Engine
→ Completeness Engine
→ Adaptive Questions
→ Human Verification
→ Human-Verified Incident Card
→ Actionability Triage
→ Operator Dashboard / Follow-up

### Safety principles

- Confidence score **bukan satu-satunya dasar keputusan**.
- V3A `NON_EVENT` tanpa dukungan `FALSE-EVENT` dari SRL **tidak langsung menolak laporan**.
- Konflik V3A vs V2 → `UNCERTAIN`.
- Impact adalah suggestion; human confirmation tetap diperlukan.
- Victim information selalu memerlukan human verification.
- Prediksi death-victim yang berada dalam pola negasi seperti *“tidak ada korban meninggal”* disuppress oleh deterministic negation guard.
- `UNCERTAIN` adalah **state produk**, bukan label neural model.

> Aturan pada notebook ini harus dikunci **sebelum** New Blind Direct-Report Test.

## Technology Reference Lock

**AI/NLP**
- Backbone: `indolem/indobert-base-uncased`
- Event model: Hugging Face `AutoModelForSequenceClassification`
- SRL model: Hugging Face `AutoModelForTokenClassification`
- Runtime: PyTorch
- Framework: Hugging Face Transformers

**Evaluation / Processing**
- scikit-learn
- pandas
- Google Colab GPU

**MVP Web**
- Django
- SQLite
- deterministic Fusion / Completeness Engine
- human-in-the-loop verification

**Dataset**
- Semantic Role Labeling Datasets for Crisis Event (Mendeley Data)

V3A baseline dipakai; V3A-W disimpan hanya sebagai experimental artifact.

## 1. Safe Colab setup

In [ ]:
!nvidia-smi
!pip install -q transformers pandas

import os
import re
import json
import math
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 78)
print("TANGGAPMAKNA V3 — FUSION ENGINE")
print("=" * 78)
print("Device :", DEVICE)
print("PyTorch:", torch.__version__)

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU belum aktif. Pilih Runtime > Change runtime type > GPU."
    )

## 2. Upload TWO selected model checkpoints

Upload sekaligus:
1. `TanggapMakna_IndoBERT_V2.zip`
2. `TanggapMakna_V3A_EventClassifier.zip`

**Jangan upload V3A-W sebagai classifier final.**

In [ ]:
from google.colab import files

uploaded = files.upload()

uploaded_zips = [
    Path("/content") / name
    for name in uploaded.keys()
    if name.lower().endswith(".zip")
]

if len(uploaded_zips) != 2:
    raise RuntimeError(
        "Upload tepat dua ZIP: V2 SRL dan V3A Event Classifier."
    )

for p in uploaded_zips:
    print(
        p.name,
        f"{p.stat().st_size/(1024**2):.2f} MB"
    )

## 3. Extract dan identifikasi checkpoint berdasarkan ontology

In [ ]:
EXTRACT_ROOT = Path("/content/tanggapmakna_v3_fusion_models")

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)


def extract_model_zip(zip_path, target):
    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(target)

    model_dirs = []

    for cfg in target.rglob("config.json"):
        parent = cfg.parent

        has_weights = (
            (parent / "model.safetensors").exists()
            or (parent / "pytorch_model.bin").exists()
        )

        if has_weights:
            model_dirs.append(parent)

    if len(model_dirs) != 1:
        raise RuntimeError(
            f"{zip_path.name}: expected 1 checkpoint directory, "
            f"ditemukan {len(model_dirs)}."
        )

    return model_dirs[0]


extracted = []

for i, zip_path in enumerate(uploaded_zips):
    model_dir = extract_model_zip(
        zip_path,
        EXTRACT_ROOT / f"model_{i}",
    )

    config = json.loads(
        (model_dir / "config.json").read_text(
            encoding="utf-8"
        )
    )

    labels = set(
        str(v)
        for v in config.get("id2label", {}).values()
    )

    extracted.append({
        "zip": zip_path,
        "dir": model_dir,
        "labels": labels,
    })


EXPECTED_SRL = {
    "O",
    "PLACE-ARG",
    "FALSE-EVENT",
    "STREET-ARG",
    "AFFECTEDOBJECTS-ARG",
    "TIME-ARG",
    "OFFICER-ARG",
    "INFORMATION-ARG",
    "FIRE-EVENT",
    "REASON-ARG",
    "ACCIDENT-EVENT",
    "DEATHVICTIM-ARG",
    "EARTHQUAKE-EVENT",
    "FLOOD-EVENT",
    "WOUNDVICTIM-ARG",
}

EXPECTED_EVENT = {
    "FLOOD",
    "EARTHQUAKE",
    "FIRE",
    "ACCIDENT",
    "NON_EVENT",
}

SRL_MODEL_DIR = None
EVENT_MODEL_DIR = None

for item in extracted:
    print("\nZIP :", item["zip"].name)
    print("DIR :", item["dir"])
    print("LABELS:", sorted(item["labels"]))

    if item["labels"] == EXPECTED_SRL:
        SRL_MODEL_DIR = item["dir"]

    elif item["labels"] == EXPECTED_EVENT:
        EVENT_MODEL_DIR = item["dir"]


if SRL_MODEL_DIR is None:
    raise RuntimeError(
        "V2 SRL checkpoint tidak ditemukan."
    )

if EVENT_MODEL_DIR is None:
    raise RuntimeError(
        "V3A baseline Event Classifier tidak ditemukan."
    )

print("\n✅ V2 SRL :", SRL_MODEL_DIR)
print("✅ V3A    :", EVENT_MODEL_DIR)

## 4. Load both selected models

In [ ]:
srl_tokenizer = AutoTokenizer.from_pretrained(
    SRL_MODEL_DIR,
    local_files_only=True,
)

srl_model = AutoModelForTokenClassification.from_pretrained(
    SRL_MODEL_DIR,
    local_files_only=True,
)

event_tokenizer = AutoTokenizer.from_pretrained(
    EVENT_MODEL_DIR,
    local_files_only=True,
)

event_model = AutoModelForSequenceClassification.from_pretrained(
    EVENT_MODEL_DIR,
    local_files_only=True,
)

srl_model.to(DEVICE)
event_model.to(DEVICE)

srl_model.eval()
event_model.eval()


def normalize_id2label(config):
    return {
        int(k): str(v)
        for k, v in config.id2label.items()
    }


SRL_ID2LABEL = normalize_id2label(
    srl_model.config
)

EVENT_ID2LABEL = normalize_id2label(
    event_model.config
)

assert set(SRL_ID2LABEL.values()) == EXPECTED_SRL
assert set(EVENT_ID2LABEL.values()) == EXPECTED_EVENT

print("SRL model   :", srl_model.__class__.__name__)
print("Event model :", event_model.__class__.__name__)
print("✅ Selected checkpoints loaded successfully.")

## 5. Locked Fusion Configuration

Threshold di sini **bukan auto-verification threshold**.

`classifier_support_min_confidence = 0.60` hanya menentukan apakah classifier-only event candidate cukup layak untuk **pre-fill yang wajib dikonfirmasi manusia**.

Tidak ada kondisi pada V3 yang melewati Human Verification.

In [ ]:
FUSION_CONFIG = {
    "version": "TanggapMakna_V3_Fusion_v1.1",
    "classifier_support_min_confidence": 0.60,
    "confidence_high": 0.80,
    "confidence_medium": 0.60,
    "support_roles": [
        "PLACE-ARG",
        "STREET-ARG",
        "TIME-ARG",
        "AFFECTEDOBJECTS-ARG",
        "INFORMATION-ARG",
        "REASON-ARG",
        "OFFICER-ARG",
        "DEATHVICTIM-ARG",
        "WOUNDVICTIM-ARG",
    ],
    "human_verification_required": True,
    "impact_confirmation_required": True,
    "victim_confirmation_required": True,
    "non_event_requires_false_event_support": True,
    "blind_test_locked_after_this_notebook": True,
}

print(
    json.dumps(
        FUSION_CONFIG,
        indent=2,
        ensure_ascii=False,
    )
)

## 6. V3A sentence-level inference

In [ ]:
MAX_LENGTH = 256


@torch.inference_mode()
def predict_event_classifier(text):
    encoded = event_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    encoded = {
        k: v.to(DEVICE)
        for k, v in encoded.items()
    }

    logits = event_model(**encoded).logits[0]
    probs = torch.softmax(logits, dim=-1)

    pred_id = int(probs.argmax().item())

    prob_dict = {
        EVENT_ID2LABEL[i]: float(probs[i].item())
        for i in range(len(probs))
    }

    return {
        "label": EVENT_ID2LABEL[pred_id],
        "confidence": float(probs[pred_id].item()),
        "probabilities": prob_dict,
    }

## 7. V2 SRL token-level inference

Inference dibuat konsisten dengan V2:
- tokenisasi laporan menjadi word/punctuation;
- WordPiece;
- label diambil dari subword pertama.

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\w+(?:[-']\w+)*|[^\w\s]",
    flags=re.UNICODE,
)


def tokenize_with_offsets(text):
    result = []

    for match in TOKEN_PATTERN.finditer(text):
        result.append({
            "token": match.group(0).lower(),
            "original": match.group(0),
            "start": match.start(),
            "end": match.end(),
        })

    return result


def srl_encode_words(tokens):
    input_ids = [srl_tokenizer.cls_token_id]
    positions = []

    for word in tokens:
        pieces = srl_tokenizer.tokenize(str(word))

        if not pieces:
            pieces = [srl_tokenizer.unk_token]

        piece_ids = srl_tokenizer.convert_tokens_to_ids(pieces)

        remaining = MAX_LENGTH - 1 - len(input_ids)

        if remaining <= 0:
            break

        piece_ids = piece_ids[:remaining]
        positions.append(len(input_ids))
        input_ids.extend(piece_ids)

    input_ids.append(srl_tokenizer.sep_token_id)

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "positions": positions,
    }


@torch.inference_mode()
def predict_srl_tokens(text):
    token_items = tokenize_with_offsets(text)
    tokens = [x["token"] for x in token_items]

    encoded = srl_encode_words(tokens)

    inputs = {
        "input_ids": torch.tensor(
            [encoded["input_ids"]],
            dtype=torch.long,
            device=DEVICE,
        ),
        "attention_mask": torch.tensor(
            [encoded["attention_mask"]],
            dtype=torch.long,
            device=DEVICE,
        ),
    }

    logits = srl_model(**inputs).logits[0]
    probs = torch.softmax(logits, dim=-1)

    output = []

    for i, pos in enumerate(encoded["positions"]):
        pred_id = int(probs[pos].argmax().item())

        item = token_items[i].copy()
        item["label"] = SRL_ID2LABEL[pred_id]
        item["confidence"] = float(
            probs[pos, pred_id].item()
        )

        output.append(item)

    return output

## 8. Convert SRL tokens to semantic spans

In [ ]:
def contiguous_role_spans(pred_tokens, role):
    spans = []
    current = []

    def flush():
        nonlocal current

        if current:
            spans.append({
                "role": role,
                "text": " ".join(
                    x["original"]
                    for x in current
                ),
                "mean_confidence": float(
                    np.mean([
                        x["confidence"]
                        for x in current
                    ])
                ),
            })

        current = []

    for item in pred_tokens:
        if item["label"] == role:
            current.append(item)
        else:
            flush()

    flush()
    return spans


def all_srl_spans(pred_tokens):
    roles = sorted(EXPECTED_SRL - {"O"})

    return {
        role: contiguous_role_spans(
            pred_tokens,
            role,
        )
        for role in roles
    }

## 9. Deterministic Time Fallback

Ini **bukan model AI tambahan**.

Fallback hanya digunakan bila V2 SRL tidak menemukan `TIME-ARG`.

In [ ]:
TIME_PATTERNS = [
    r"\b(?:pukul|jam)\s+\d{1,2}(?:[.:]\d{2})?(?:\s*(?:pagi|siang|sore|malam))?\b",
    r"\b(?:tadi|sejak)\s+(?:pagi|siang|sore|malam)\b",
    r"\b(?:pagi|siang|sore|malam)\s+ini\b",
    r"\bmalam\s+tadi\b",
    r"\bkemarin\b",
    r"\bbarusan\b",
    r"\btadi\b",
    r"\bhari\s+ini\b",
]


def time_fallback(text):
    matches = []
    occupied = []

    for pattern in TIME_PATTERNS:
        for m in re.finditer(
            pattern,
            text,
            flags=re.IGNORECASE,
        ):
            start, end = m.span()

            overlaps = any(
                start < e and end > s
                for s, e in occupied
            )

            if overlaps:
                continue

            matches.append({
                "role": "TIME-FALLBACK",
                "text": m.group(0),
                "source": "DETERMINISTIC_REGEX",
            })

            occupied.append((start, end))

    return sorted(
        matches,
        key=lambda x: text.lower().find(
            x["text"].lower()
        ),
    )

## 10. Death-victim Negation Guard

Jika teks memuat negasi eksplisit seperti:
- `tidak ada korban meninggal`
- `tidak terdapat korban jiwa`
- `nihil korban jiwa`
- `tanpa korban meninggal`

maka kandidat `DEATHVICTIM-ARG` dari model disuppress.

In [ ]:

DEATH_NEGATION_PATTERNS = [
    r"\btidak\s+ada\s+(?:korban\s+)?(?:meninggal|korban\s+jiwa)\b",
    r"\btidak\s+terdapat\s+(?:korban\s+)?(?:meninggal|korban\s+jiwa)\b",
    r"\bnihil\s+(?:korban\s+)?(?:meninggal|korban\s+jiwa)\b",
    r"\btanpa\s+(?:korban\s+)?(?:meninggal|korban\s+jiwa)\b",
]

DEATH_EVIDENCE_PATTERNS = [
    r"\bmeninggal\b",
    r"\btewas\b",
    r"\bwafat\b",
    r"\bmenewaskan\b",
    r"\bkorban\s+jiwa\b",
]

WOUND_EVIDENCE_PATTERNS = [
    r"\bterluka\b",
    r"\bluka\b",
    r"\bluka\s+bakar\b",
    r"\bcedera\b",
    r"\bmengalami\s+luka\b",
    r"\bmengeluh\s+sakit\b",
]


def has_any_pattern(text, patterns):
    return any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in patterns
    )


def has_death_negation(text):
    return has_any_pattern(text, DEATH_NEGATION_PATTERNS)


def normalize_victim_phrase(text, kind):
    number_word = (
        r"(?:\d+|satu|dua|tiga|empat|lima|enam|tujuh|"
        r"delapan|sembilan|sepuluh)"
    )
    person_word = r"(?:orang|korban|pengendara|penumpang|warga)"

    if kind == "wound":
        cue = (
            r"(?:terluka|cedera|luka\s+bakar|"
            r"mengalami\s+luka|mengeluh\s+sakit)"
        )
    else:
        cue = r"(?:meninggal|tewas|wafat)"

    pattern = (
        rf"\b(?:{number_word}\s+)?"
        rf"(?:{person_word}\s+)?"
        rf"{cue}\b"
    )

    match = re.search(pattern, text, flags=re.IGNORECASE)
    return match.group(0) if match else None


def apply_victim_safety(text, spans):
    safety_flags = []

    death_spans = spans.get("DEATHVICTIM-ARG", [])
    wound_spans = spans.get("WOUNDVICTIM-ARG", [])

    death_negated = has_death_negation(text)
    death_evidence = has_any_pattern(
        text,
        DEATH_EVIDENCE_PATTERNS,
    )
    wound_evidence = has_any_pattern(
        text,
        WOUND_EVIDENCE_PATTERNS,
    )

    if death_spans and death_negated:
        spans["DEATHVICTIM-ARG"] = []
        safety_flags.append(
            "DEATH_VICTIM_SUPPRESSED_BY_NEGATION"
        )

    elif death_spans and not death_evidence:
        spans["DEATHVICTIM-ARG"] = []
        safety_flags.append(
            "DEATH_VICTIM_SUPPRESSED_NO_DEATH_EVIDENCE"
        )

    elif death_spans and death_evidence:
        phrase = normalize_victim_phrase(text, "death")
        if phrase:
            spans["DEATHVICTIM-ARG"] = [{
                "role": "DEATHVICTIM-ARG",
                "text": phrase,
                "mean_confidence": float(np.mean([
                    x.get("mean_confidence", 0.0)
                    for x in death_spans
                ])),
                "source": "SRL_PLUS_LEXICAL_EVIDENCE",
            }]

    if wound_spans and not wound_evidence:
        spans["WOUNDVICTIM-ARG"] = []
        safety_flags.append(
            "WOUND_VICTIM_SUPPRESSED_NO_WOUND_EVIDENCE"
        )

    elif wound_spans and wound_evidence:
        phrase = normalize_victim_phrase(text, "wound")
        if phrase:
            spans["WOUNDVICTIM-ARG"] = [{
                "role": "WOUNDVICTIM-ARG",
                "text": phrase,
                "mean_confidence": float(np.mean([
                    x.get("mean_confidence", 0.0)
                    for x in wound_spans
                ])),
                "source": "SRL_PLUS_LEXICAL_EVIDENCE",
            }]

    return spans, safety_flags


## 10A. Explicit Ambiguity Guard — Fusion v1.1

Jika pelapor sendiri menyatakan ketidakpastian secara eksplisit,
neural agreement tidak boleh mengubahnya menjadi kejadian terkonfirmasi.


In [ ]:

EXPLICIT_AMBIGUITY_PATTERNS = [
    r"\bbelum\s+diketahui\s+apakah\b",
    r"\bbelum\s+tahu\s+apakah\b",
    r"\bbelum\s+jelas\s+apakah\b",
    r"\bsaya\s+tidak\s+yakin\b",
    r"\btidak\s+yakin\s+(?:itu|apakah)\b",
    r"\b(?:bisa|mungkin)\s+.+?\s+atau\s+.+",
]

HEARSAY_PATTERNS = [
    r"\bkatanya\b",
    r"\bkabarnya\b",
    r"\bmenurut\s+kabar\b",
    r"\bdikabarkan\b",
]


def detect_explicit_uncertainty(text):
    reasons = []

    if any(
        re.search(p, text, flags=re.IGNORECASE)
        for p in EXPLICIT_AMBIGUITY_PATTERNS
    ):
        reasons.append("EXPLICIT_AMBIGUITY")

    if any(
        re.search(p, text, flags=re.IGNORECASE)
        for p in HEARSAY_PATTERNS
    ):
        reasons.append("HEARSAY")

    return {
        "is_uncertain": bool(reasons),
        "reasons": reasons,
    }


## 11. Locked Fusion / Consistency Rules

### Rule hierarchy

**A. Strong Actual Event Agreement**
- V3A predicts event E
- V2 contains exactly the same true-event E
- no `FALSE-EVENT`
→ `ACTUAL_EVENT` / `AGREEMENT_STRONG`

**B. Strong False Event Agreement**
- V3A = `NON_EVENT`
- V2 contains `FALSE-EVENT`
- no true event
→ `FALSE_EVENT` / `NON_EVENT_AGREEMENT`

**C. Classifier-Supported Implicit Event**
- V3A predicts event E
- V2 has no true/false event
- V2 extracts ≥1 supporting semantic role
- classifier confidence ≥0.60
→ `ACTUAL_EVENT` / `CLASSIFIER_SUPPORTED`
→ mandatory human confirmation

**D. Any disagreement/conflict**
→ `UNCERTAIN` / `CONFLICT`

**E. Classifier-only NON_EVENT**
→ `UNCERTAIN`
→ never auto-reject report

**F. Weak classifier-only event**
→ `UNCERTAIN`

In [ ]:
SRL_TRUE_EVENT_TO_CLASS = {
    "FLOOD-EVENT": "FLOOD",
    "EARTHQUAKE-EVENT": "EARTHQUAKE",
    "FIRE-EVENT": "FIRE",
    "ACCIDENT-EVENT": "ACCIDENT",
}

SUPPORT_ROLES = set(FUSION_CONFIG["support_roles"])


def confidence_band(conf):
    if conf >= FUSION_CONFIG["confidence_high"]:
        return "HIGH"

    if conf >= FUSION_CONFIG["confidence_medium"]:
        return "MEDIUM"

    return "LOW"


def derive_srl_signals(spans):
    true_events = sorted({
        SRL_TRUE_EVENT_TO_CLASS[role]
        for role in SRL_TRUE_EVENT_TO_CLASS
        if spans.get(role)
    })

    has_false = bool(spans.get("FALSE-EVENT"))

    support_present = {
        role: spans.get(role, [])
        for role in SUPPORT_ROLES
        if spans.get(role)
    }

    return {
        "true_events": true_events,
        "has_false": has_false,
        "support_present": support_present,
        "support_role_count": len(support_present),
    }


def fusion_decision(event_pred, srl_spans, uncertainty_signal=None):
    event_label = event_pred["label"]
    event_conf = event_pred["confidence"]
    signals = derive_srl_signals(srl_spans)

    true_events = signals["true_events"]
    has_false = signals["has_false"]
    support_count = signals["support_role_count"]

    result = {
        "Event_Status": "UNCERTAIN",
        "Event_Type": "UNCERTAIN",
        "Fusion_State": "UNCERTAIN",
        "Fusion_Reason": "",
        "Classifier_Label": event_label,
        "Classifier_Confidence": event_conf,
        "Classifier_Confidence_Band": confidence_band(
            event_conf
        ),
        "SRL_True_Events": true_events,
        "SRL_Has_False_Event": has_false,
        "SRL_Support_Role_Count": support_count,
        "Human_Verification_Required": True,
    }

    # Safety override: explicit uncertainty from reporter
    # outranks neural agreement.
    if uncertainty_signal and uncertainty_signal.get("is_uncertain"):
        result.update({
            "Fusion_State": "UNCERTAIN",
            "Fusion_Reason": (
                "EXPLICIT_REPORTER_UNCERTAINTY:"
                + ",".join(
                    uncertainty_signal.get("reasons", [])
                )
            ),
        })
        return result

    if (
        len(true_events) > 1
        or (true_events and has_false)
    ):
        result.update({
            "Fusion_State": "CONFLICT",
            "Fusion_Reason": "SRL_EVENT_CONFLICT",
        })
        return result

    if event_label == "NON_EVENT":
        if has_false and not true_events:
            result.update({
                "Event_Status": "FALSE_EVENT",
                "Event_Type": "NONE",
                "Fusion_State": "NON_EVENT_AGREEMENT",
                "Fusion_Reason": (
                    "V3A_NON_EVENT_AND_V2_FALSE_EVENT"
                ),
            })
            return result

        if true_events:
            result.update({
                "Fusion_State": "CONFLICT",
                "Fusion_Reason": (
                    "V3A_NON_EVENT_VS_V2_TRUE_EVENT"
                ),
            })
            return result

        result.update({
            "Fusion_State": "UNCERTAIN",
            "Fusion_Reason": (
                "CLASSIFIER_ONLY_NON_EVENT"
            ),
        })
        return result

    if true_events:
        v2_event = true_events[0]

        if v2_event == event_label:
            result.update({
                "Event_Status": "ACTUAL_EVENT",
                "Event_Type": event_label,
                "Fusion_State": "AGREEMENT_STRONG",
                "Fusion_Reason": (
                    "V3A_AND_V2_EVENT_AGREE"
                ),
            })
            return result

        result.update({
            "Fusion_State": "CONFLICT",
            "Fusion_Reason": (
                "V3A_V2_EVENT_TYPE_DISAGREE"
            ),
        })
        return result

    if has_false:
        result.update({
            "Fusion_State": "CONFLICT",
            "Fusion_Reason": (
                "V3A_EVENT_VS_V2_FALSE_EVENT"
            ),
        })
        return result

    if (
        support_count >= 1
        and event_conf
        >= FUSION_CONFIG[
            "classifier_support_min_confidence"
        ]
    ):
        result.update({
            "Event_Status": "ACTUAL_EVENT",
            "Event_Type": event_label,
            "Fusion_State": "CLASSIFIER_SUPPORTED",
            "Fusion_Reason": (
                "V3A_EVENT_WITH_SUPPORTING_SRL_ARGUMENTS"
            ),
        })
        return result

    result.update({
        "Fusion_State": "UNCERTAIN",
        "Fusion_Reason": (
            "WEAK_CLASSIFIER_ONLY_EVENT"
        ),
    })

    return result

## 12. Incident Card builder

In [ ]:

def normalize_display_text(text):
    text = str(text)

    text = re.sub(
        r"\s+([,;:!?])",
        r"\1",
        text,
    )

    text = re.sub(
        r"(?<=\d)\s*\.\s*(?=\d)",
        ".",
        text,
    )

    text = re.sub(
        r"\s+\.",
        ".",
        text,
    )

    text = re.sub(
        r"\s{2,}",
        " ",
        text,
    )

    return text.strip()


def span_texts(spans, role):
    return [
        normalize_display_text(x["text"])
        for x in spans.get(role, [])
    ]


def build_incident_card(text, fusion, spans):
    time_values = span_texts(
        spans,
        "TIME-ARG",
    )

    time_source = "SRL"

    if not time_values:
        fallback = time_fallback(text)

        time_values = [
            normalize_display_text(x["text"])
            for x in fallback
        ]

        time_source = (
            "DETERMINISTIC_FALLBACK"
            if time_values
            else "MISSING"
        )

    return {
        "event_status": fusion["Event_Status"],
        "event_type": fusion["Event_Type"],
        "place": span_texts(spans, "PLACE-ARG"),
        "street": span_texts(spans, "STREET-ARG"),
        "time": time_values,
        "time_source": time_source,
        "affected_objects": span_texts(
            spans,
            "AFFECTEDOBJECTS-ARG",
        ),
        "information_impact": span_texts(
            spans,
            "INFORMATION-ARG",
        ),
        "reason": span_texts(spans, "REASON-ARG"),
        "officer": span_texts(spans, "OFFICER-ARG"),
        "death_victim": span_texts(
            spans,
            "DEATHVICTIM-ARG",
        ),
        "wound_victim": span_texts(
            spans,
            "WOUNDVICTIM-ARG",
        ),
        "fusion_state": fusion["Fusion_State"],
        "human_verified": False,
    }


## 13. Completeness Engine + Adaptive Questions

Completeness Engine bersifat deterministic dan tidak mengarang field yang tidak ditemukan.

In [ ]:
def completeness_engine(card, safety_flags):
    questions = []
    missing_fields = []

    if card["event_status"] == "FALSE_EVENT":
        return {
            "missing_fields": [],
            "adaptive_questions": [
                (
                    "Sistem mendeteksi bahwa teks mungkin bukan laporan "
                    "kejadian aktual. Apakah klasifikasi ini benar?"
                )
            ],
            "requires_human_verification": True,
            "actionability_triage": (
                "PENDING_HUMAN_VERIFICATION"
            ),
            "safety_flags": safety_flags,
        }

    if (
        card["event_status"] == "UNCERTAIN"
        or card["event_type"] == "UNCERTAIN"
    ):
        missing_fields.append("event_confirmation")
        questions.append(
            "Jenis kejadian apa yang sedang dilaporkan?"
        )

    if not (card["place"] or card["street"]):
        missing_fields.append("location")
        questions.append(
            "Di mana lokasi kejadian atau kondisi ini diamati?"
        )

    if not card["time"]:
        missing_fields.append("time")
        questions.append(
            "Kapan kejadian atau kondisi ini diamati?"
        )

    if not (
        card["affected_objects"]
        or card["information_impact"]
    ):
        missing_fields.append("impact")
        questions.append(
            "Apa dampak yang terlihat atau objek apa yang terdampak?"
        )

    if (
        card["death_victim"]
        or card["wound_victim"]
    ):
        questions.append(
            "Sistem mendeteksi informasi korban. "
            "Mohon konfirmasi apakah informasi korban tersebut benar."
        )
    else:
        missing_fields.append("victim_status")
        questions.append(
            "Apakah terdapat korban? "
            "Pilih Ya, Tidak, atau Tidak tahu."
        )

    return {
        "missing_fields": missing_fields,
        "adaptive_questions": questions,
        "requires_human_verification": True,
        "actionability_triage": (
            "PENDING_HUMAN_VERIFICATION"
        ),
        "safety_flags": safety_flags,
    }

## 14. End-to-end Fusion function

In [ ]:
def analyze_report(text):
    event_pred = predict_event_classifier(text)
    srl_tokens = predict_srl_tokens(text)
    spans = all_srl_spans(srl_tokens)

    spans, safety_flags = apply_victim_safety(
        text,
        spans,
    )

    uncertainty_signal = detect_explicit_uncertainty(
        text
    )

    fusion = fusion_decision(
        event_pred,
        spans,
        uncertainty_signal=uncertainty_signal,
    )

    if uncertainty_signal["is_uncertain"]:
        safety_flags.append(
            "EXPLICIT_REPORTER_UNCERTAINTY"
        )

    if fusion["Fusion_State"] == "CONFLICT":
        safety_flags.append(
            "MODEL_EVENT_CONFLICT"
        )

    card = build_incident_card(
        text,
        fusion,
        spans,
    )

    completeness = completeness_engine(
        card,
        safety_flags,
    )

    return {
        "report_text": text,
        "event_classifier": event_pred,
        "srl_spans": spans,
        "fusion": fusion,
        "incident_card": card,
        "completeness": completeness,
        "srl_token_predictions": srl_tokens,
    }

## Fusion v1.1 QA Expectations

Sebelum blind test:

- D10 harus menjadi `UNCERTAIN`.
- D05: death victim palsu `orang` harus hilang; wound menjadi frasa utuh seperti `dua orang terluka`.
- D06: death candidate tetap disuppress karena negasi.
- D11: `dibawa ambulans` tidak boleh otomatis menjadi wound victim.
- D01/D02/D04/D07/D08/D09/D12 tidak boleh rusak.


## 15. Diagnostic reports

**Ini bukan blind final test.**

Tujuan:
- memastikan fusion code berjalan;
- memeriksa behavior;
- memeriksa safety cases sebelum rule dikunci.

In [ ]:
DIAGNOSTIC_REPORTS = [
    "Banjir terjadi di Sukarame dan merendam beberapa rumah sejak pukul 15.00.",
    "Air naik cepat di Way Halim dan dua rumah sudah terendam sejak sore.",
    "Bangunan bergoyang dan warga langsung keluar rumah.",
    "Kebakaran terjadi di kios dekat pasar sekitar pukul 21.15.",
    "Kecelakaan terjadi di Jalan Soekarno Hatta dan menyebabkan dua orang terluka.",
    "Mobil terguling di Jalan Pangeran Antasari, tidak ada korban meninggal.",
    "Pemerintah sedang membahas solusi banjir untuk tahun depan.",
    "Sekolah mengadakan simulasi evakuasi kebakaran pagi ini.",
    "Video gempa yang beredar itu ternyata rekaman lama dan bukan kejadian hari ini.",
    "Asap terlihat dari arah belakang pasar, belum diketahui apakah kebakaran atau pembakaran sampah.",
    "Ada informasi dua orang dibawa ambulans dari jalan depan mall, penyebabnya belum diketahui.",
    "Gempa terasa di Kemiling malam tadi dan warga keluar rumah.",
]

diagnostic_results = []

for i, text in enumerate(
    DIAGNOSTIC_REPORTS,
    start=1,
):
    result = analyze_report(text)

    fusion = result["fusion"]
    card = result["incident_card"]
    comp = result["completeness"]

    diagnostic_results.append({
        "id": f"D{i:02d}",
        "text": text,
        "classifier": result[
            "event_classifier"
        ]["label"],
        "classifier_conf": round(
            result["event_classifier"][
                "confidence"
            ],
            4,
        ),
        "srl_true_events": ", ".join(
            fusion["SRL_True_Events"]
        ),
        "srl_false": fusion[
            "SRL_Has_False_Event"
        ],
        "event_status": card[
            "event_status"
        ],
        "event_type": card[
            "event_type"
        ],
        "fusion_state": card[
            "fusion_state"
        ],
        "place": "; ".join(card["place"]),
        "street": "; ".join(card["street"]),
        "time": "; ".join(card["time"]),
        "time_source": card["time_source"],
        "affected": "; ".join(
            card["affected_objects"]
        ),
        "impact": "; ".join(
            card["information_impact"]
        ),
        "death_victim": "; ".join(
            card["death_victim"]
        ),
        "wound_victim": "; ".join(
            card["wound_victim"]
        ),
        "missing": "; ".join(
            comp["missing_fields"]
        ),
        "safety_flags": "; ".join(
            comp["safety_flags"]
        ),
    })

diagnostic_df = pd.DataFrame(
    diagnostic_results
)

pd.set_option(
    "display.max_colwidth",
    120,
)

display(diagnostic_df)

## 16. Detailed diagnostic inspection

In [ ]:
for i, text in enumerate(
    DIAGNOSTIC_REPORTS,
    start=1,
):
    result = analyze_report(text)

    print("\n" + "=" * 100)
    print(f"D{i:02d}")
    print(text)

    print("\nEVENT CLASSIFIER")
    print(
        json.dumps(
            result["event_classifier"],
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\nFUSION")
    print(
        json.dumps(
            result["fusion"],
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\nINCIDENT CARD")
    print(
        json.dumps(
            result["incident_card"],
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\nCOMPLETENESS")
    print(
        json.dumps(
            result["completeness"],
            indent=2,
            ensure_ascii=False,
        )
    )

## 17. Lock Fusion Rules / Export Configuration

Jika Cell 15–16 secara teknis berjalan dan tidak ada bug implementasi, configuration ini dikunci untuk New Blind Test.

Perubahan setelah blind-test results dianggap iteration baru dan membutuhkan test set baru.

In [ ]:
LOCK_DIR = Path(
    "/content/TanggapMakna_V3_Fusion_Locked"
)

LOCK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

fusion_manifest = {
    "project": "TanggapMakna",
    "architecture_version": "V3 Fusion v1.1",
    "selected_models": {
        "event_classifier": (
            "TanggapMakna V3A baseline — "
            "indolem/indobert-base-uncased "
            "sequence classification"
        ),
        "srl_extractor": (
            "TanggapMakna V2 — "
            "indolem/indobert-base-uncased "
            "token classification / SRL"
        ),
        "v3aw_status": (
            "Rejected as final classifier; "
            "experimental artifact only"
        ),
    },
    "fusion_config": FUSION_CONFIG,
    "safety_rules": {
        "classifier_non_event_alone_cannot_auto_reject": True,
        "model_conflict_yields_uncertain": True,
        "impact_requires_confirmation": True,
        "victim_requires_confirmation": True,
        "death_negation_guard": True,
        "actionability_before_human_verification": False,
    },
    "time_fallback_patterns": TIME_PATTERNS,
    "death_negation_patterns": DEATH_NEGATION_PATTERNS,
    "next_evaluation": (
        "New blind direct-report test "
        "created after fusion lock"
    ),
    "old_local_60_status": (
        "Diagnostic only for V2; "
        "must not be reused as blind V3 benchmark"
    ),
}

manifest_path = (
    LOCK_DIR
    / "TanggapMakna_V3_Fusion_Manifest.json"
)

manifest_path.write_text(
    json.dumps(
        fusion_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

diagnostic_df.to_csv(
    LOCK_DIR
    / "Fusion_Diagnostic_Results.csv",
    index=False,
)

archive = shutil.make_archive(
    "/content/TanggapMakna_V3_Fusion_Locked",
    "zip",
    root_dir=LOCK_DIR,
)

print("Manifest:", manifest_path)
print("Archive :", archive)
print("\n🔒 Fusion rules ready to lock before blind test.")

# ⛔ FUSION CHECKPOINT

Kirim ke ChatGPT:

1. **Cell 15** — tabel `diagnostic_df`.
2. Bila ada hasil yang terlihat aneh, kirim bagian terkait dari **Cell 16**.
3. Pastikan Cell 17 berhasil membuat:
   `TanggapMakna_V3_Fusion_Locked.zip`

## Setelah Fusion technically passes

`LOCK FUSION`
→ **create NEW blind direct-report set**
→ human ground truth
→ run V3 Fusion ONCE
→ final metrics
→ freeze AI
→ Django web implementation

**Jangan gunakan 60 Local Challenge Set lama sebagai final V3 benchmark.**

## 18. Optional download Fusion Manifest

In [ ]:
from google.colab import files

files.download(
    "/content/TanggapMakna_V3_Fusion_Locked.zip"
)